In [0]:
from pyspark.sql.functions import col

quality_df = spark.table("workspace.default.gold_quality_by_table")
rule_df = spark.table("workspace.default.gold_quality_by_rule")

def get_worst_quality_table():
    return quality_df.orderBy(col("quality_score").asc()).limit(1).collect()[0]

def get_failed_rules(table_name):
    return (
        rule_df
        .filter(col("table_name") == table_name)
        .orderBy(col("total_failure_percentage").desc())
        .collect()
    )

def build_agent_response(user_question):
    worst = get_worst_quality_table()
    table_name = worst["table_name"]
    failed_rules = get_failed_rules(table_name)

    response = f"""
Question:
{user_question}

Answer:
The table with the worst data quality is `{table_name}`.

Reason:
It has the lowest quality score of {worst["quality_score"]}.
It has {worst["failed_rules"]} failed rules out of {worst["total_rules"]} total rules.

Top failed rule areas:
"""

    for rule in failed_rules:
        response += f"""
- Rule: {rule["rule"]}
  Total Failure Percentage: {rule["total_failure_percentage"]}
"""

    response += """

Recommended remediation:
- Add validation checks earlier in the Bronze/Silver layer.
- Add deduplication for uniqueness failures.
- Add mandatory field checks for not-null failures.
- Track failed rules historically to detect recurring issues.
- Add alerting when quality score drops below threshold.

Agent design:
This response was generated by calling Databricks Delta-backed tools:
1. get_worst_quality_table()
2. get_failed_rules()
3. generate_remediation_plan()
"""

    return response

question = "Which table has the worst data quality and what should we do?"
print(build_agent_response(question))